In [4]:
import pandas as pd
from pathlib import Path


In [ ]:
# """
# Merge all DataFrame CSVs into a single dataset
# """

# path = "/data/elugos/sbert_title_121225"
# path = Path(path)

# csv_files = list()
# for dir, _, files in path.walk():
#     for f in files:
#         if Path(f).suffix == '.csv':
#             csv_files.append(dir.joinpath(f))

# print(f"Found {len(csv_files)} files.")

In [ ]:
# import pandas as pd

# df_list = [pd.read_csv(f) for f in csv_files]
# df = pd.concat(df_list)

# df.head()

In [ ]:
# output_file = "/data/elugos/event_embedding/train.csv"
# df.to_csv(output_file, index=None)

In [ ]:
# import re

# regex = re.compile(r"(.+){30,}\n")
# # print(df['text'][0])

# match = regex.findall(df['text'][0])
# if match:
#     print(match)


In [325]:
import pandas as pd
import re

def extract_first_paragraphs(df: pd.DataFrame, col: str='text', min_len: int=40):
    """
    Extract the first meaningful paragraph from a dataframe column containing article text.

    Heuristics:
      - Prefer chunks separated by blank lines.
      - Skip leading short/byline/date-like chunks.
      - A 'real' paragraph should be >= min_len OR contain sentence-ending punctuation.
      - Fallback: if no blank lines exist, assemble from lines after skipping short/noisy headers.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    col : str
        Column name containing the article text.
    min_len : int
        Minimum length threshold for a chunk to be considered a paragraph.
    
    Returns
    -------
    pandas.Series
        A series containing the extracted first paragraph for each row (or None if not found).
    """

    # Sentence end detection (loose heuristic)
    # sentence_end_re = re.compile(r'[.!?]["\')]*\s|[.!?]["\')]*$')
    sentence_end_re = re.compile(r"(.+)[\n$]")

    # Patterns commonly seen in noisy headers/bylines/dates (non-exhaustive)
    noisy_line_res = [
        re.compile(r'^\s*(?:by|author|reporting|editor|correspondent)\b', re.I),  # byline
        re.compile(r'^\s*(?:updated|published|last\s+modified)\b', re.I),         # metadata
        re.compile(r'^\s*\(?\d{1,2}\s+\w+\s+\d{4}\)?$', re.I),                    # "14 Dec 2025"
        re.compile(r'^\s*\w+\s+\d{1,2},\s+\d{4}\s*$', re.I),                      # "December 14, 2025"
        re.compile(r'^\s*(AP|Reuters|AFP)\b', re.I),                               # wire service tags
        re.compile(r'^\s*(Photo|Image|Credit)\b', re.I),                           # media credits
        re.compile(r'^[A-Z \-]{6,}$'),                                             # all-caps headers
    ]


    def looks_noisy(line: str) -> bool:
        s = line.strip()
        if not s:
            return True
        # Very short lines without sentence punctuation
        if len(s) < min_len and not sentence_end_re.search(s):
            return True
        # Known noisy patterns
        for rx in noisy_line_res:
            if rx.search(s):
                return True
        return False

    def is_paragraph_candidate(chunk: str) -> bool:
        s = chunk.strip()
        if not s:
            return False
        # Consider a chunk a paragraph if it's long OR has sentence-ending punctuation
        return len(s) >= min_len or bool(sentence_end_re.search(s))

    def extract(text):
        if not isinstance(text, str):
            return None

        # Normalize whitespace
        s = text.strip()
        if not s:
            return None

        # First pass: split by blank lines (common paragraph boundary)
        paragraphs = [p.strip() for p in re.split(r'\n\s*\n', s) if p.strip()]

        # Skip initial noisy chunks
        for p in paragraphs:
            if is_paragraph_candidate(p) and not looks_noisy(p):
                return p

        # Fallback: build from individual lines (when no blank lines or all chunks seemed noisy)
        lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
        # Skip leading noisy lines
        i = 0
        while i < len(lines) and looks_noisy(lines[i]):
            i += 1

        if i >= len(lines):
            return None

        # Accumulate lines until we hit an empty separator (rare here since empties were removed)
        # or until we have a candidate paragraph.
        acc = []
        for j in range(i, len(lines)):
            acc.append(lines[j])
            para = ' '.join(acc)
            if is_paragraph_candidate(para):
                return para

        # If nothing matched the heuristics, return the remaining accumulated text (best effort)
        return ' '.join(acc) if acc else None

    return df['text'].apply(extract)

In [326]:
import pandas as pd
from pathlib import Path
"""
Merge DataFrames per event. E.g.: all files in 2024_Helene_FL will be merged into 2024_Helene_FL.csv
"""

input_path = Path("/data/elugos/event_data/goosed")
event_dirs = [d for d in input_path.iterdir() if d.is_dir()]

for path in event_dirs:
    print(f"> {path.name}")
    output_file = input_path.parent.joinpath(f"raw/{path.name}.csv")
    csv_files = list()
    if output_file.exists():
        print(f"    File {output_file} already exists. Skipping merge.")
        continue
    for dir, subdirs, files in path.walk():
        if dir.name == 'process':  # Only crawl if we are within a 'process' folder
            for f in files:
                if Path(f).suffix == '.csv':
                    csv_files.append(dir.joinpath(f))

    print(f"     Found {len(csv_files)} files.")
    df_list = [pd.read_csv(f) for f in csv_files]
    df = pd.concat(d for d in df_list if len(d)>0)
    if len(df) > 0:
        df = df.sort_values(by='date')  # Save DFs sorted by date
        df['first_para'] = extract_first_paragraphs(df, col='text')
        df.to_csv(output_file, index=None)
        print(f"     Saved {output_file}")



> 2024_vehicleram_CA
    File /data/elugos/event_data/raw/2024_vehicleram_CA.csv already exists. Skipping merge.
> 2019_vabeach_VA
    File /data/elugos/event_data/raw/2019_vabeach_VA.csv already exists. Skipping merge.
> 2025_fires_CA
    File /data/elugos/event_data/raw/2025_fires_CA.csv already exists. Skipping merge.
> 2024_Helene_FL
    File /data/elugos/event_data/raw/2024_Helene_FL.csv already exists. Skipping merge.
> 2023_Lewiston_MA
    File /data/elugos/event_data/raw/2023_Lewiston_MA.csv already exists. Skipping merge.
> 2021_tornado_kentucky
     Found 40 files.
     Saved /data/elugos/event_data/raw/2021_tornado_kentucky.csv
> 2023_monterey_ca
    File /data/elugos/event_data/raw/2023_monterey_ca.csv already exists. Skipping merge.
> 2025_floods_TX
    File /data/elugos/event_data/raw/2025_floods_TX.csv already exists. Skipping merge.
> 2019_elpaso_tx
    File /data/elugos/event_data/raw/2019_elpaso_tx.csv already exists. Skipping merge.
> 2022_massshooting_NY
    File /d